In [ ]:
import warnings
from pathlib import Path

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

# --- Configurazione centralizzata -------------------------------------------
RANDOM_STATE = 42                 # seed unico per tutta la pipeline -> riproducibilità
# --- Sorgenti dati: online su GitHub + Kaggle (con fallback locale) ----------
# Il dataset viene risolto automaticamente in base all'ambiente di esecuzione:
#   1) su Kaggle si usa il dataset gia' montato in /kaggle/input (nessun accesso
#      a internet -> nessun errore/warning nei log);
#   2) altrove (locale, Colab, ...) si scarica il CSV "raw" da GitHub;
#   3) in ultima istanza si usa una copia locale, se presente.
DATA_FILENAME = "AmesHousing.csv"

# Kaggle: cartella in cui il dataset viene montato una volta aggiunto al notebook
# https://www.kaggle.com/datasets/shashanknecrothapa/ames-housing-dataset
KAGGLE_DATASET_DIR = "/kaggle/input/ames-housing-dataset"

# GitHub: URL "raw" del CSV (2930 righe x 82 colonne, con le stringhe "NA"
# preservate come "caratteristica assente" -> identico all'originale di De Cock
# e al file su Kaggle).
GITHUB_CSV_URL = (
    "https://raw.githubusercontent.com/josephpconley/R/master/"
    "openintrostat/OpenIntroLabs/(4)%20lab4/data%20%26%20custom%20code/AmesHousing.csv"
)

# Fallback locale (se il CSV e' nella cartella di lavoro).
LOCAL_CSV_PATH = Path(DATA_FILENAME)
TARGET = "SalePrice"
ID_COLS = ["Order", "PID"]        # identificatori: non predittivi, da escludere dal modello

# --- Impostazioni di visualizzazione ----------------------------------------
pd.set_option("display.max_columns", 120)
pd.set_option("display.width", 160)
sns.set_theme(style="whitegrid", palette="deep")
plt.rcParams["figure.dpi"] = 110
warnings.filterwarnings("ignore", category=FutureWarning)

np.random.seed(RANDOM_STATE)
print("Ambiente configurato. Seed =", RANDOM_STATE)

In [ ]:
def _resolve_data_source():
    """Individua la sorgente del dataset in base all'ambiente.

    Ordine di priorita':
      1) Kaggle  -> file gia' montato in /kaggle/input (offline: nessun log error)
      2) locale  -> copia nella cartella di lavoro
      3) GitHub  -> download del CSV "raw" (richiede connessione)

    Returns
    -------
    (source, origin) : tuple
        `source` e' un percorso o URL passabile a pd.read_csv;
        `origin` e' un'etichetta leggibile della sorgente scelta.
    """
    import glob

    # 1) Kaggle: cerca AmesHousing.csv ovunque dentro /kaggle/input
    kaggle_hits = []
    if Path("/kaggle/input").exists():
        kaggle_hits += glob.glob(f"/kaggle/input/**/{DATA_FILENAME}", recursive=True)
    kaggle_hits += glob.glob(str(Path(KAGGLE_DATASET_DIR) / "**" / DATA_FILENAME), recursive=True)
    if kaggle_hits:
        return kaggle_hits[0], "Kaggle (dataset montato, offline)"

    # 2) copia locale nella cartella di lavoro
    if LOCAL_CSV_PATH.exists():
        return str(LOCAL_CSV_PATH), "file locale"

    # 3) GitHub (online)
    return GITHUB_CSV_URL, "GitHub (online)"


def load_data(source=None, *, raw_na: bool = True) -> pd.DataFrame:
    """Carica il dataset Ames Housing in modo riproducibile e portabile.

    La sorgente viene scelta automaticamente: Kaggle se il notebook gira su
    Kaggle, altrimenti GitHub (o una copia locale). Si puo' forzare passando
    `source` (percorso o URL).

    Parameters
    ----------
    source : str | None
        Percorso/URL esplicito. Se None la sorgente e' risolta automaticamente.
    raw_na : bool, default True
        Se True conserva le stringhe "NA" originali (significato semantico:
        "caratteristica assente"). Se False lascia che pandas le converta in NaN.

    Returns
    -------
    pd.DataFrame
        Il dataset grezzo, con le sole pulizie strutturali sicure (rimozione
        spazi nei nomi colonna, trim delle stringhe, rimozione di eventuali
        colonne-indice spurie).
    """
    if source is None:
        source, origin = _resolve_data_source()
    else:
        origin = "sorgente specificata"

    # keep_default_na=False -> "NA" resta stringa, non diventa NaN automaticamente.
    # na_values=[""] -> solo le celle realmente vuote diventano NaN.
    read_kwargs = dict(keep_default_na=False, na_values=[""]) if raw_na else {}
    try:
        df = pd.read_csv(source, **read_kwargs)
    except Exception as exc:
        raise RuntimeError(
            f"Impossibile caricare il dataset da '{source}' ({origin}).\n"
            "- Su Kaggle: aggiungi il dataset "
            "'shashanknecrothapa/ames-housing-dataset' al notebook (Add Input).\n"
            "- In locale/Colab: verifica la connessione a internet oppure copia "
            f"'{DATA_FILENAME}' nella cartella di lavoro."
        ) from exc

    # Pulizie strutturali sicure (non alterano i valori, solo la forma):
    df.columns = df.columns.str.strip()                        # nomi colonna senza spazi
    df = df.loc[:, ~df.columns.str.contains(r"^Unnamed")]      # via colonne-indice spurie
    obj_cols = df.select_dtypes(include=["object", "string"]).columns
    df[obj_cols] = df[obj_cols].apply(lambda s: s.str.strip())  # trim valori testuali

    print(f"Sorgente dati: {origin}")
    print(f"  -> {source}")
    return df


# Caricamento (sorgente risolta automaticamente: Kaggle / GitHub / locale)
df = load_data()
print(f"Dataset caricato correttamente: {df.shape[0]} righe x {df.shape[1]} colonne")
df.head()

In [ ]:
def dataset_overview(df: pd.DataFrame) -> None:
    """Stampa un riepilogo strutturale ad alto livello del DataFrame."""
    n_rows, n_cols = df.shape
    
    # Seleziona i tipi numerici
    num_cols = df.select_dtypes(include=np.number).columns
    
    # CORREZIONE: Include sia 'object' che 'string' per retrocompatibilità e silenziare il warning
    cat_cols = df.select_dtypes(include=["object", "string", "str"]).columns
    
    mem_mb = df.memory_usage(deep=True).sum() / 1024**2

    print("=" * 55)
    print(f"{'Osservazioni (righe)':<35}{n_rows:>20}")
    print(f"{'Variabili (colonne)':<35}{n_cols:>20}")
    print(f"{'  - numeriche':<35}{len(num_cols):>20}")
    print(f"{'  - categoriche / testuali':<35}{len(cat_cols):>20}")
    print(f"{'Variabile target':<35}{TARGET:>20}")
    print(f"{'Occupazione in memoria (MB)':<35}{mem_mb:>20.2f}")
    print("=" * 55)


dataset_overview(df)





In [ ]:
print("Conteggio per tipo di dato:")
print(df.dtypes.value_counts().to_string())

print("\nColonne identificative (da escludere dal modello):")
for c in ID_COLS:
    print(f"  - {c}: {df[c].nunique()} valori unici su {len(df)} righe")

In [ ]:
def build_data_dictionary(df: pd.DataFrame) -> pd.DataFrame:
    """Genera un profilo sintetico per ogni colonna del DataFrame.

    Per ogni variabile riporta: tipo, numero di valori distinti,
    conteggio e percentuale di valori mancanti (NaN reali) e
    un campione di valori osservati.
    """
    profile = pd.DataFrame({
        "dtype": df.dtypes.astype(str),
        "n_unique": df.nunique(dropna=True),
        "n_missing": df.isna().sum(),
    })
    profile["pct_missing"] = (profile["n_missing"] / len(df) * 100).round(2)
    profile["sample_values"] = [
    ", ".join(map(str, df[c].dropna().unique()[:4]))[:50] + "..." 
    if len(", ".join(map(str, df[c].dropna().unique()[:4]))) > 50 
    else ", ".join(map(str, df[c].dropna().unique()[:4]))
    for c in df.columns
]

    return profile.sort_values("pct_missing", ascending=False)


data_dict = build_data_dictionary(df)
print(f"Dizionario dati: {data_dict.shape[0]} variabili profilate.")
data_dict.head(20)

In [ ]:
# 1) NaN reali (celle vuote) -- questi sono i veri mancanti da gestire
real_missing = df.isna().sum()
real_missing = real_missing[real_missing > 0].sort_values(ascending=False)

print(f"Colonne con NaN REALI (celle vuote): {len(real_missing)}")
print(real_missing.to_string())

# 2) Stringa "NA" semantica ("feature assente"): conteggio per colonna
na_semantic = (df == "NA").sum()
na_semantic = na_semantic[na_semantic > 0].sort_values(ascending=False)

print(f"\nColonne dove \"NA\" significa FEATURE ASSENTE (non un dato mancante): {len(na_semantic)}")
print(na_semantic.head(12).to_string())

In [ ]:



def impute_real_missing_values(df: pd.DataFrame) -> pd.DataFrame:
    """Imputa i veri valori mancanti (NaN) nel dataset Ames Housing

    in modo differenziato e sicuro, restituendo una copia del DataFrame.
    """
    # Lavoriamo su una copia per evitare modifiche distruttive sul DataFrame originale
    df_clean = df.copy()

    # 1. Lot Frontage (distanza dalla strada):
    # Spesso dipende fortemente dal quartiere. Se un quartiere ha lotti grandi,
    # la distanza sarà simile. Usiamo la mediana specifica del quartiere ('Neighborhood').
    if "Lot Frontage" in df_clean.columns and "Neighborhood" in df_clean.columns:
        df_clean["Lot Frontage"] = df_clean.groupby("Neighborhood")[
            "Lot Frontage"
        ].transform(lambda x: x.fillna(x.median()))

        # Fallback di sicurezza: se un intero quartiere non ha valori, usa la mediana globale
        global_lot_median = df_clean["Lot Frontage"].median()
        df_clean["Lot Frontage"] = df_clean["Lot Frontage"].fillna(
            global_lot_median
        )

    # 2. Garage Yr Blt (Anno di costruzione del garage):
    # Se manca, quasi sempre significa che il garage non esiste.
    # Sostituire con 0 o la mediana altererebbe i modelli lineari o ad albero.
    # La scelta migliore è allinearlo all'anno di costruzione della casa ('Year Built').
    if "Garage Yr Blt" in df_clean.columns and "Year Built" in df_clean.columns:
        df_clean["Garage Yr Blt"] = df_clean["Garage Yr Blt"].fillna(
            df_clean["Year Built"]
        )

    # 3. Variabili numeriche legate a componenti aggiuntive (Muratura, Bagni, Superfici extra):
    # Se mancano, significa che l'estensione ha superficie zero o il conteggio è zero.
    zero_impute_cols = [
        "Mas Vnr Area",  # Superficie rivestimento in muratura
        "BsmtFin SF 1",
        "BsmtFin SF 2",
        "Bsmt Unf SF",
        "Total Bsmt SF",  # Superfici del seminterrato
        "Bsmt Full Bath",
        "Bsmt Half Bath",  # Bagni del seminterrato
        "Garage Cars",
        "Garage Area",  # Capacità e superficie del garage
    ]

    for col in zero_impute_cols:
        if col in df_clean.columns:
            df_clean[col] = df_clean[col].fillna(0.0)

    # 4. Variabili categoriche residue (es. Mas Vnr Type se presente come vero NaN, Electrical, ecc.):
    # Per le stringhe residue che non sono state catturate dal controllo "NA",
    # inseriamo la moda della colonna (il valore più frequente) o una stringa "Unknown".
    categorical_cols = df_clean.select_dtypes(
        include=["object", "string", "str"]
    ).columns
    for col in categorical_cols:
        # Se ci sono veri NaN rimasti nelle colonne testuali
        if df_clean[col].isna().any():
            # Electrical ha tipicamente 1 solo valore mancante, usiamo la moda
            most_frequent = df_clean[col].mode()[0]
            df_clean[col] = df_clean[col].fillna(most_frequent)

    return df_clean


# --- Applicazione della pipeline ---
# Assicurati che 'df' sia caricato
df_imputed = impute_real_missing_values(df)

# Controllo finale di verifica
nan_rimasti = df_imputed.isna().sum().sum()
print(f"Imputazione completata.")
print(f"Numero totale di veri NaN rimasti nel dataset: {nan_rimasti}")


In [ ]:


# Supponiamo che tu stia usando il DataFrame 'df_imputed' dallo step precedente


def handle_outliers(df: pd.DataFrame, target_col: str = "SalePrice") -> pd.DataFrame:
    """Mostra gli outlier identificati e restituisce un DataFrame pulito

    secondo le raccomandazioni ufficiali di Dean De Cock (Gr Liv Area <= 4000).
    """
    # 1. Identificazione visiva degli outlier raccomandati
    outliers_mask = df["Gr Liv Area"] > 4000
    n_outliers = outliers_mask.sum()

    print("=" * 60)
    print(f"OUTLIER DETECTED (Gr Liv Area > 4000 piedi quadrati)")
    print(f"Numero di osservazioni da rimuovere: {n_outliers}")
    print("=" * 60)

    if n_outliers > 0:
        print(
            df[outliers_mask][["Gr Liv Area", target_col]].to_string(index=False)
        )
    else:
        print("Nessun outlier trovato sopra i 4000 mq (forse già rimossi?).")

    # 2. Rimozione degli outlier
    df_cleaned = df[~outliers_mask].copy()

    print(f"\nDimensioni dataset ORIGINALE: {df.shape}")
    print(f"Dimensioni dataset PULITO:    {df_cleaned.shape}")
    print("=" * 60)

    return df_cleaned


# Applichiamo la funzione
df_cleaned = handle_outliers(df_imputed)


In [ ]:
# Creiamo un confronto grafico Prima vs Dopo
fig, axes = plt.subplots(1, 2, figsize=(14, 5), sharey=True)

# Grafico 1: Dataset Originale con Outlier evidenti
sns.scatterplot(
    data=df_imputed,
    x="Gr Liv Area",
    y="SalePrice",
    alpha=0.6,
    ax=axes[0],
    color="darkred",
)
# Evidenziamo la linea di soglia dei 4000 sq ft
axes[0].axvline(
    x=4000,
    color="black",
    linestyle="--",
    alpha=0.7,
    label="Soglia De Cock (4000 sq ft)",
)
axes[0].set_title("Prima della pulizia (Con Outlier)", fontsize=12)
axes[0].set_xlabel("Gr Liv Area (Superficie Abitabile)")
axes[0].set_ylabel("SalePrice (Prezzo di Vendita)")
axes[0].legend()

# Grafico 2: Dataset Pulito
sns.scatterplot(
    data=df_cleaned,
    x="Gr Liv Area",
    y="SalePrice",
    alpha=0.6,
    ax=axes[1],
    color="teal",
)
axes[1].set_title("Dopo la pulizia (Senza Outlier)", fontsize=12)
axes[1].set_xlabel("Gr Liv Area (Superficie Abitabile)")
axes[1].set_ylabel("")  # Rimosso perché condiviso con il primo asse

plt.tight_layout()
plt.show()


In [ ]:
# Le statistiche del target vanno calcolate sui dati PULITI (senza outlier)
price = df_cleaned[TARGET]

stats = {
    "Conteggio": price.count(),
    "Media": price.mean(),
    "Mediana": price.median(),
    "Dev. standard": price.std(),
    "Minimo": price.min(),
    "Massimo": price.max(),
    "Asimmetria (skew)": price.skew(),
    "Curtosi": price.kurt(),
}
for k, v in stats.items():
    print(f"{k:<20}{v:>15,.2f}")

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(13, 4.5))

# Distribuzione originale: chiaramente asimmetrica a destra
sns.histplot(price, kde=True, ax=axes[0], color="#2c7fb8")
axes[0].set_title(f"SalePrice — scala originale (skew = {price.skew():.2f})")
axes[0].set_xlabel("Prezzo di vendita ($)")

# Distribuzione log: si avvicina molto a una normale
sns.histplot(np.log1p(price), kde=True, ax=axes[1], color="#31a354")
axes[1].set_title(f"log(1 + SalePrice) — skew = {np.log1p(price).skew():.2f}")
axes[1].set_xlabel("log(1 + Prezzo)")

plt.tight_layout()
plt.show()

In [ ]:
# Alcune feature numeriche fra le più rilevanti per il prezzo di un immobile
key_numeric = ["Gr Liv Area", "Total Bsmt SF", "Garage Area",
               "Lot Area", "Year Built", "Overall Qual"]
key_numeric = [c for c in key_numeric if c in df_cleaned.columns]

fig, axes = plt.subplots(2, 3, figsize=(14, 7))
for ax, col in zip(axes.ravel(), key_numeric):
    sns.histplot(df_cleaned[col], kde=True, ax=ax, color="#756bb1")
    ax.set_title(col)
    ax.set_xlabel("")
plt.tight_layout()
plt.show()

In [ ]:


# CORREZIONE: Include sia 'object' che 'string' per retrocompatibilità e silenziare il warning
cat_cols = df_cleaned.select_dtypes(include=["object", "string"]).columns

# Calcolo della cardinalità
cardinality = df_cleaned[cat_cols].nunique().sort_values(ascending=False)

print("Variabili categoriche per numero di categorie distinte:")
print(cardinality.to_string())
print(f"\nTotale variabili categoriche: {len(cat_cols)}")
print(
    f"Cardinalità media: {cardinality.mean():.1f} | massima: {cardinality.max()} "
    f"({cardinality.idxmax()})"
)


In [ ]:
# --- Categoria A: "NA" categorico = FEATURE ASSENTE -> diventa categoria "None" ----
# (es. Pool QC=NA -> nessuna piscina). Fix deterministico, nessun leakage.
SEMANTIC_NONE_CAT = [
    "Alley", "Bsmt Qual", "Bsmt Cond", "Bsmt Exposure",
    "BsmtFin Type 1", "BsmtFin Type 2", "Fireplace Qu",
    "Garage Type", "Garage Finish", "Garage Qual", "Garage Cond",
    "Pool QC", "Fence", "Misc Feature", "Mas Vnr Type",
]

# --- Categoria B: numeriche STRUTTURALMENTE assenti -> 0 ---------------------------
# (nessun garage/seminterrato => area, bagni, posti auto = 0). Deterministico.
STRUCTURAL_ZERO_NUM = [
    "Mas Vnr Area", "BsmtFin SF 1", "BsmtFin SF 2", "Bsmt Unf SF",
    "Total Bsmt SF", "Bsmt Full Bath", "Bsmt Half Bath",
    "Garage Cars", "Garage Area",
]

# --- Categoria C: mancanti VERI da imputare statisticamente ------------------------
#   Lot Frontage -> mediana per quartiere (Neighborhood): cattura la regolarità urbana
#   Electrical   -> moda (1 solo mancante)
#   Garage Yr Blt-> trattamento dedicato (vedi sotto)
STAT_IMPUTE = ["Lot Frontage", "Electrical", "Garage Yr Blt"]

print("Categoria A (NA semantico -> 'None'):", len(SEMANTIC_NONE_CAT), "colonne")
print("Categoria B (strutturale -> 0):     ", len(STRUCTURAL_ZERO_NUM), "colonne")
print("Categoria C (imputazione statistica):", len(STAT_IMPUTE), "colonne")

In [ ]:
# === FASE 2 — Pipeline corretta (anti-leakage) ===
from sklearn.model_selection import train_test_split

# 1) Dati grezzi: le stringhe "NA" semantiche sono ancora intatte (load_data, raw_na=True)
df_raw = load_data()

# 2) Outlier deterministici (De Cock): Gr Liv Area > 4000
n_before = len(df_raw)
df_raw = df_raw[df_raw["Gr Liv Area"] <= 4000].copy()
print(f"Outlier rimossi (Gr Liv Area > 4000): {n_before - len(df_raw)}  ->  righe: {len(df_raw)}")

# 3) Target in scala log (coerente con l'analisi della distribuzione in Fase 1)
X = df_raw.drop(columns=[TARGET] + ID_COLS)
y = np.log1p(df_raw[TARGET])

# 4) SPLIT prima di qualsiasi statistica -> nessun data leakage
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=RANDOM_STATE
)
print(f"Train: {X_train.shape[0]} righe | Test: {X_test.shape[0]} righe")

In [ ]:
def fix_deterministic(df: pd.DataFrame) -> pd.DataFrame:
    """Fix deterministici: nessuna statistica fra le righe, nessun uso del target.

    Sicuri da applicare a qualunque porzione di dati (train o test).
    """
    df = df.copy()
    # Categoria A: "NA" semantico (e eventuali NaN reali) -> categoria esplicita "None"
    #   .replace cattura la stringa "NA"; .fillna cattura i NaN reali (es. Mas Vnr Type)
    for col in SEMANTIC_NONE_CAT:
        df[col] = df[col].replace("NA", "None").fillna("None")
    # Categoria B: assenza strutturale (no garage/seminterrato/...) -> 0
    for col in STRUCTURAL_ZERO_NUM:
        df[col] = df[col].fillna(0.0)
    return df


# --- Categoria C: parametri APPRESI SOLO SUL TRAIN -------------------------------
neigh_lotfront_median = X_train.groupby("Neighborhood")["Lot Frontage"].median()
global_lotfront_median = X_train["Lot Frontage"].median()
electrical_mode = X_train["Electrical"].mode()[0]


def apply_statistical(df: pd.DataFrame) -> pd.DataFrame:
    """Imputazione statistica con i parametri del TRAIN. Usare dopo fix_deterministic."""
    df = df.copy()
    # Lot Frontage -> mediana del quartiere (fallback: mediana globale del train)
    df["Lot Frontage"] = df["Lot Frontage"].fillna(df["Neighborhood"].map(neigh_lotfront_median))
    df["Lot Frontage"] = df["Lot Frontage"].fillna(global_lotfront_median)
    # Electrical -> moda del train (1 solo mancante)
    df["Electrical"] = df["Electrical"].fillna(electrical_mode)
    # Garage Yr Blt -> anno di costruzione casa (row-wise, deterministico)
    df["Garage Yr Blt"] = df["Garage Yr Blt"].fillna(df["Year Built"])
    return df

In [ ]:
# Applicazione: stessi parametri (stimati sul train) applicati a train E test
X_train = apply_statistical(fix_deterministic(X_train))
X_test  = apply_statistical(fix_deterministic(X_test))

# Verifica: 0 NaN residui da entrambe le parti
nan_train = int(X_train.isna().sum().sum())
nan_test  = int(X_test.isna().sum().sum())
print(f"NaN residui  ->  train: {nan_train} | test: {nan_test}")
assert nan_train == 0 and nan_test == 0, "Mancanti ancora presenti: verifica le liste A/B/C"

print("Pipeline Fase 2 completata senza data leakage.")
print(f"Shape finali ->  X_train {X_train.shape} | X_test {X_test.shape}")

In [ ]:
from sklearn.base import BaseEstimator, TransformerMixin
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import OrdinalEncoder, OneHotEncoder, StandardScaler, FunctionTransformer

DERIVED = ["House Age", "Years Since Remod", "Total SF", "Total Bath",
           "Total Porch SF", "Overall Score", "SF per Room"]

# Colonne di portici/terrazzi: assenza = 0 piedi quadri (non "sconosciuto").
# Esplicitiamo .fillna(0) prima di sommare cosi l'intento e' auto-documentante e
# robusto anche se in futuro queste colonne arrivassero con dei NaN.
PORCH_COLS = ["Open Porch SF", "Enclosed Porch", "3Ssn Porch",
              "Screen Porch", "Wood Deck SF"]

def add_derived_features(df: pd.DataFrame) -> pd.DataFrame:
    """Crea feature derivate (deterministiche, riga-per-riga -> nessun leakage)."""
    df = df.copy()
    df["House Age"]         = (df["Yr Sold"] - df["Year Built"]).clip(lower=0)
    df["Years Since Remod"] = (df["Yr Sold"] - df["Year Remod/Add"]).clip(lower=0)
    df["Total SF"]          = df["Total Bsmt SF"] + df["1st Flr SF"] + df["2nd Flr SF"]
    df["Total Bath"]        = (df["Full Bath"] + 0.5 * df["Half Bath"]
                               + df["Bsmt Full Bath"] + 0.5 * df["Bsmt Half Bath"])
    df["Total Porch SF"]    = df[PORCH_COLS].fillna(0).sum(axis=1)
    df["Overall Score"]     = df["Overall Qual"] * df["Overall Cond"]
    df["SF per Room"]       = df["Gr Liv Area"] / df["TotRms AbvGrd"].clip(lower=1)
    df["MS SubClass"]       = df["MS SubClass"].astype(str)   # codice numerico ma categoriale -> nominale
    return df

In [ ]:
QUAL = ["None", "Po", "Fa", "TA", "Gd", "Ex"]
ORDINAL_ORDERS = {
    "Exter Qual": QUAL, "Exter Cond": QUAL, "Bsmt Qual": QUAL, "Bsmt Cond": QUAL,
    "Heating QC": QUAL, "Kitchen Qual": QUAL, "Fireplace Qu": QUAL,
    "Garage Qual": QUAL, "Garage Cond": QUAL, "Pool QC": QUAL,
    "Bsmt Exposure":  ["None", "No", "Mn", "Av", "Gd"],
    "BsmtFin Type 1": ["None", "Unf", "LwQ", "Rec", "BLQ", "ALQ", "GLQ"],
    "BsmtFin Type 2": ["None", "Unf", "LwQ", "Rec", "BLQ", "ALQ", "GLQ"],
    "Garage Finish":  ["None", "Unf", "RFn", "Fin"],
    "Functional":     ["Sal", "Sev", "Maj2", "Maj1", "Mod", "Min2", "Min1", "Typ"],
    "Lot Shape":      ["IR3", "IR2", "IR1", "Reg"],
    "Land Slope":     ["Sev", "Mod", "Gtl"],
    "Paved Drive":    ["N", "P", "Y"],
    "Utilities":      ["NoSeWa", "NoSewr", "AllPub"],
    "Fence":          ["None", "MnWw", "GdWo", "MnPrv", "GdPrv"],
    "Central Air":    ["N", "Y"],
    "Street":         ["Grvl", "Pave"],
    "Electrical":     ["Mix", "FuseP", "FuseF", "FuseA", "SBrkr"],
}
ORDINAL_COLS = list(ORDINAL_ORDERS.keys())
NOMINAL_COLS = ["MS SubClass", "MS Zoning", "Alley", "Land Contour", "Lot Config",
                "Neighborhood", "Condition 1", "Condition 2", "Bldg Type", "House Style",
                "Roof Style", "Roof Matl", "Exterior 1st", "Exterior 2nd", "Mas Vnr Type",
                "Foundation", "Heating", "Garage Type", "Misc Feature",
                "Sale Type", "Sale Condition"]

# Protezione difensiva: identificativi e target non devono MAI finire fra le feature.
# (Sono gia esclusi in Fase 2 costruendo X per sottrazione, ma qui ci proteggiamo
#  esplicitamente nel caso l'ordine delle fasi cambiasse o i dati venissero ricaricati.)
LEAK_COLS = [TARGET] + ID_COLS   # SalePrice, Order, PID
assert not (set(LEAK_COLS) & set(X_train.columns)), (
    f"Identificativi/target ancora dentro X_train: {set(LEAK_COLS) & set(X_train.columns)}"
)

# Le numeriche sono tutto cio che resta dopo aver creato le derivate
# (escludendo ordinali, nominali e, per sicurezza, eventuali colonne di leakage)
_cols_after_derive = add_derived_features(X_train).columns
NUMERIC_COLS = [c for c in _cols_after_derive
                if c not in ORDINAL_COLS + NOMINAL_COLS + LEAK_COLS]
print(f"Numeriche: {len(NUMERIC_COLS)} | Ordinali: {len(ORDINAL_COLS)} | Nominali: {len(NOMINAL_COLS)}")
print(f"Guardia anti-leakage OK: nessuno fra {LEAK_COLS} presente in X_train.")

In [ ]:
def _require_named_columns(X, where):
    """Guard: garantisce che X arrivi con nomi di colonna reali (DataFrame con header
    testuali). Se le colonne sono indici interi significa che a monte manca
    set_output(transform="pandas"): meglio fallire subito e chiaro che degradare
    in silenzio perdendo i nomi delle feature."""
    X = pd.DataFrame(X)
    if pd.api.types.is_integer_dtype(X.columns):
        raise ValueError(
            f"{where}: colonne senza nome (indici interi {list(X.columns[:3])}...). "
            "Manca set_output(transform='pandas') a monte: aggiungilo al Pipeline / "
            "ColumnTransformer cosi i nomi delle feature vengono preservati."
        )
    return X


class SkewedLogTransformer(BaseEstimator, TransformerMixin):
    """log1p CONDIZIONATO: comprime solo le colonne non-negative con |skew| > soglia,
    e SOLO se il log riduce davvero l'asimmetria (guard anti-overcorrection).

    Le colonne da comprimere sono decise in `fit` (quindi sul solo training set).
    Il guard evita casi come le variabili "zero-inflated" (molti zeri + coda a destra),
    dove il log1p puo' ribaltare la distribuzione e peggiorare lo skew.
    """
    def __init__(self, threshold: float = 0.75):
        self.threshold = threshold

    def fit(self, X, y=None):
        X = _require_named_columns(X, "SkewedLogTransformer")
        skew = X.apply(lambda s: s.skew())
        candidates = [c for c in X.columns
                      if abs(skew[c]) > self.threshold and X[c].min() >= 0]
        # guard: tieni il log solo dove riduce effettivamente |skew|
        self.cols_to_log_ = [c for c in candidates
                             if abs(np.log1p(X[c]).skew()) < abs(skew[c])]
        self.feature_names_in_ = np.array(X.columns)
        return self

    def transform(self, X):
        X = pd.DataFrame(X).copy()
        for c in self.cols_to_log_:
            X[c] = np.log1p(X[c])
        return X

    def get_feature_names_out(self, input_features=None):
        return self.feature_names_in_


class DropDuplicateColumns(BaseEstimator, TransformerMixin):
    """Rimuove le colonne perfettamente duplicate (collinearita' perfetta).

    Le colonne da scartare sono individuate in `fit` (sul training set), mantenendo
    la prima occorrenza, e rimosse identicamente da train/test/fold -> nessun
    disallineamento. Esempio tipico su Ames: MS SubClass_90 == Bldg Type_Duplex.
    """
    def fit(self, X, y=None):
        X = _require_named_columns(X, "DropDuplicateColumns")
        self.cols_to_drop_ = X.columns[X.T.duplicated().values].tolist()
        self.feature_names_in_ = np.array(X.columns)
        return self

    def transform(self, X):
        X = pd.DataFrame(X)
        return X.drop(columns=self.cols_to_drop_, errors="ignore")

    def get_feature_names_out(self, input_features=None):
        return np.array([c for c in self.feature_names_in_ if c not in self.cols_to_drop_])

In [ ]:
num_pipe = Pipeline([
    ("skewlog", SkewedLogTransformer(threshold=0.75)),
    ("scale",   StandardScaler()),
])
ordinal_encoder = OrdinalEncoder(
    categories=[ORDINAL_ORDERS[c] for c in ORDINAL_COLS],
    handle_unknown="use_encoded_value", unknown_value=-1,
)
nominal_encoder = OneHotEncoder(handle_unknown="ignore", sparse_output=False, min_frequency=10)

column_transformer = ColumnTransformer(
    transformers=[
        ("num", num_pipe,        NUMERIC_COLS),
        ("ord", ordinal_encoder, ORDINAL_COLS),
        ("nom", nominal_encoder, NOMINAL_COLS),
    ],
    remainder="drop",
    verbose_feature_names_out=False,
)

preprocessor = Pipeline([
    ("derive", FunctionTransformer(add_derived_features)),
    ("ct",     column_transformer),
    ("dedup",  DropDuplicateColumns()),   # rimuove le dummy perfettamente duplicate
])
preprocessor.set_output(transform="pandas")

# Addestramento SOLO sul train, applicazione a train e test
X_train_prep = preprocessor.fit_transform(X_train)
X_test_prep  = preprocessor.transform(X_test)

print(f"Matrice finale -> train: {X_train_prep.shape} | test: {X_test_prep.shape}")
print(f"NaN residui    -> train: {int(X_train_prep.isna().sum().sum())} "
      f"| test: {int(X_test_prep.isna().sum().sum())}")
assert list(X_train_prep.columns) == list(X_test_prep.columns), "Colonne train/test disallineate!"
logged = preprocessor.named_steps["ct"].named_transformers_["num"].named_steps["skewlog"].cols_to_log_
dropped = preprocessor.named_steps["dedup"].cols_to_drop_
print(f"Feature compresse con log (guard skew, decise sul train): {len(logged)}")
print(f"Colonne duplicate rimosse: {dropped if dropped else 'nessuna'}")

In [ ]:
from sklearn.linear_model import Ridge
from sklearn.model_selection import cross_val_score

sanity = Pipeline([("prep", preprocessor), ("model", Ridge(alpha=10.0))])
cv_rmse = -cross_val_score(sanity, X_train, y_train, cv=5,
                           scoring="neg_root_mean_squared_error")
print(f"[sanity] Ridge 5-fold CV RMSE (target log): {cv_rmse.mean():.4f} +/- {cv_rmse.std():.4f}")

In [ ]:
import time
from sklearn.model_selection import KFold, GridSearchCV, RandomizedSearchCV, cross_val_score
from sklearn.linear_model import LinearRegression, Ridge
from sklearn.ensemble import RandomForestRegressor
from xgboost import XGBRegressor

CV = KFold(n_splits=5, shuffle=True, random_state=RANDOM_STATE)

def make_pipeline(model):
    """Incapsula preprocessor (Fase 3) + modello in un'unica Pipeline.
    La CV clona la pipeline e ri-addestra il preprocessing su ogni fold -> niente leakage."""
    return Pipeline([("prep", preprocessor), ("model", model)])

baseline_models = {
    "LinearRegression": LinearRegression(),                 # baseline richiesto dal task
    "Ridge":        Ridge(alpha=10.0),                      # versione regolarizzata (piu' robusta con tante dummy)
    "RandomForest": RandomForestRegressor(n_estimators=200, random_state=RANDOM_STATE, n_jobs=-1),
    "XGBoost":      XGBRegressor(n_estimators=400, learning_rate=0.05, max_depth=4,
                                 subsample=0.8, colsample_bytree=0.8,
                                 random_state=RANDOM_STATE, n_jobs=-1, verbosity=0),
}

baseline_rmse = {}
print("Baseline 5-fold CV (RMSE su target log1p):")
for name, model in baseline_models.items():
    scores = -cross_val_score(make_pipeline(model), X_train, y_train, cv=CV,
                              scoring="neg_root_mean_squared_error", n_jobs=-1)
    baseline_rmse[name] = scores.mean()
    print(f"  {name:13s} {scores.mean():.4f} +/- {scores.std():.4f}")

print("\nNota: LinearRegression e' il baseline lineare richiesto; Ridge ne e' la versione")
print("regolarizzata, piu' stabile con le ~200 colonne one-hot, ed e' quella che portiamo al tuning.")

In [ ]:
# --- Ridge: ricerca su griglia di alpha ---
ridge_grid = {"model__alpha": [0.3, 1, 3, 10, 30, 100]}
search_ridge = GridSearchCV(make_pipeline(Ridge()), ridge_grid, cv=CV,
                            scoring="neg_root_mean_squared_error", n_jobs=-1)
search_ridge.fit(X_train, y_train)
print(f"Ridge  -> best alpha = {search_ridge.best_params_['model__alpha']}"
      f" | CV RMSE = {-search_ridge.best_score_:.4f}")

In [ ]:
# --- Random Forest: ricerca casuale (cv=3 per contenere i tempi) ---
CV3 = KFold(n_splits=3, shuffle=True, random_state=RANDOM_STATE)
rf_grid = {
    "model__n_estimators":    [150, 250],
    "model__max_depth":       [None, 15, 25],
    "model__max_features":    [0.3, 0.5, "sqrt"],
    "model__min_samples_leaf":[1, 2, 4],
}
search_rf = RandomizedSearchCV(
    make_pipeline(RandomForestRegressor(random_state=RANDOM_STATE, n_jobs=1)),
    rf_grid, n_iter=8, cv=CV3, scoring="neg_root_mean_squared_error",
    n_jobs=-1, random_state=RANDOM_STATE)
search_rf.fit(X_train, y_train)
print(f"RandomForest -> CV RMSE = {-search_rf.best_score_:.4f}")
print(f"   best params: {search_rf.best_params_}")

In [ ]:
# --- XGBoost: ricerca casuale ---
xgb_grid = {
    "model__n_estimators":     [400, 600, 800],
    "model__learning_rate":    [0.02, 0.05, 0.1],
    "model__max_depth":        [3, 4, 5],
    "model__subsample":        [0.7, 0.8, 1.0],
    "model__colsample_bytree": [0.7, 0.8, 1.0],
    "model__reg_lambda":       [1, 5, 10],
    "model__min_child_weight": [1, 3],
}
search_xgb = RandomizedSearchCV(
    make_pipeline(XGBRegressor(random_state=RANDOM_STATE, n_jobs=1, verbosity=0)),
    xgb_grid, n_iter=15, cv=CV, scoring="neg_root_mean_squared_error",
    n_jobs=-1, random_state=RANDOM_STATE)
search_xgb.fit(X_train, y_train)
print(f"XGBoost -> CV RMSE = {-search_xgb.best_score_:.4f}")
print(f"   best params: {search_xgb.best_params_}")

In [ ]:
searches = {"Ridge": search_ridge, "RandomForest": search_rf, "XGBoost": search_xgb}
tuned_rmse = {name: -s.best_score_ for name, s in searches.items()}

print(f"{'Modello':<14}{'CV baseline':>14}{'CV ottimizzato':>18}")
print("-" * 46)
for name in baseline_models:
    tuned_col = f"{tuned_rmse[name]:>18.4f}" if name in tuned_rmse else f"{'(non ottimizzato)':>18}"
    print(f"{name:<14}{baseline_rmse[name]:>14.4f}{tuned_col}")

# dizionario delle pipeline ottimizzate (la scelta finale e' nella sezione 4.4, omogenea)
tuned_models = {name: s.best_estimator_ for name, s in searches.items()}
print("\nNota: i punteggi qui sopra mescolano 3-fold (RF) e 5-fold (Ridge/XGB).")
print("La selezione finale usa una 5-fold CV omogenea (sezione 4.4).")

In [ ]:
# Ri-valutazione omogenea: stessa CV (5-fold) per tutti i modelli ottimizzati
cv5_rmse = {}
for name, est in tuned_models.items():
    scores = -cross_val_score(est, X_train, y_train, cv=CV,
                              scoring="neg_root_mean_squared_error", n_jobs=-1)
    cv5_rmse[name] = scores.mean()

print(f"{'Modello':<14}{'CV 5-fold omogenea':>22}")
print("-" * 36)
for name in tuned_models:
    print(f"{name:<14}{cv5_rmse[name]:>22.4f}")

best_name  = min(cv5_rmse, key=cv5_rmse.get)
best_model = tuned_models[best_name]
print(f"\nMiglior modello (5-fold omogenea): {best_name}  (CV RMSE {cv5_rmse[best_name]:.4f})")

In [ ]:
from sklearn.metrics import mean_squared_error, mean_absolute_error, r2_score

def evaluate(model, X, y_log):
    """Metriche in scala log e in dollari (back-transform con expm1)."""
    pred_log = model.predict(X)
    pred_d, true_d = np.expm1(pred_log), np.expm1(y_log)
    return {
        "RMSE_log": np.sqrt(mean_squared_error(y_log, pred_log)),
        "R2":       r2_score(y_log, pred_log),
        "RMSE_$":   np.sqrt(mean_squared_error(true_d, pred_d)),
        "MAE_$":    mean_absolute_error(true_d, pred_d),
        "MAPE_%":   np.mean(np.abs((true_d - pred_d) / true_d)) * 100,
    }

print("Valutazione sul TEST (mai usato finora). La selezione era gia' fatta in CV.\n")
print(f"{'Modello':<14}{'RMSE_log':>10}{'R2':>8}{'RMSE_$':>11}{'MAE_$':>10}{'MAPE_%':>9}")
print("-" * 62)
for name, est in tuned_models.items():
    m = evaluate(est, X_test, y_test)
    star = "  <- scelto" if name == best_name else ""
    print(f"{name:<14}{m['RMSE_log']:>10.4f}{m['R2']:>8.4f}{m['RMSE_$']:>11,.0f}"
          f"{m['MAE_$']:>10,.0f}{m['MAPE_%']:>9.2f}{star}")

final = evaluate(best_model, X_test, y_test)
print(f"\nModello finale: {best_name}")
print(f"  In media il modello sbaglia di ~${final['MAE_$']:,.0f} ({final['MAPE_%']:.1f}%) "
      f"su un prezzo medio di ${np.expm1(y_test).mean():,.0f}.")

In [ ]:
# Predizioni come Series allineate all'indice di X_test (evita disallineamenti posizione/etichetta)
pred_log = pd.Series(best_model.predict(X_test), index=X_test.index)
pred_d, true_d = np.expm1(pred_log), np.expm1(y_test)
resid_d = true_d - pred_d

fig, axes = plt.subplots(1, 3, figsize=(16, 4.5))
axes[0].scatter(true_d, pred_d, alpha=0.4, s=18)
lims = [true_d.min(), true_d.max()]
axes[0].plot(lims, lims, "r--", lw=1)
axes[0].set(xlabel="Prezzo reale ($)", ylabel="Prezzo previsto ($)", title="Previsto vs reale")

axes[1].scatter(pred_d, resid_d, alpha=0.4, s=18)
axes[1].axhline(0, color="r", ls="--", lw=1)
axes[1].set(xlabel="Prezzo previsto ($)", ylabel="Residuo ($)", title="Residui vs previsto")

sns.histplot(resid_d, bins=40, kde=True, ax=axes[2])
axes[2].axvline(0, color="r", ls="--", lw=1)
axes[2].set(xlabel="Residuo ($)", title="Distribuzione dei residui")
plt.tight_layout(); plt.show()

worst = resid_d.abs().sort_values(ascending=False).head(5)
print("Le 5 previsioni piu' sbagliate (in valore assoluto, $):")
for idx, err in worst.items():
    print(f"  reale ${true_d[idx]:>10,.0f} | previsto ${pred_d[idx]:>10,.0f} | errore ${err:>+10,.0f}")

In [ ]:
from sklearn.inspection import permutation_importance

# Coefficienti del modello lineare (feature standardizzate -> confrontabili)
feat_names = best_model.named_steps["prep"].named_steps["dedup"].get_feature_names_out()
linear = best_model.named_steps["model"]
coef = pd.Series(linear.coef_, index=feat_names)
top_coef = coef.reindex(coef.abs().sort_values(ascending=False).index).head(15)[::-1]

# Permutation importance sul test (model-agnostic, per feature originale)
perm = permutation_importance(best_model, X_test, y_test, n_repeats=10,
                              random_state=RANDOM_STATE,
                              scoring="neg_root_mean_squared_error", n_jobs=-1)
perm_imp = pd.Series(perm.importances_mean, index=X_test.columns).sort_values().tail(15)

fig, axes = plt.subplots(1, 2, figsize=(15, 6))
colors = ["#c0392b" if v < 0 else "#27ae60" for v in top_coef]
axes[0].barh(top_coef.index, top_coef.values, color=colors)
axes[0].axvline(0, color="k", lw=0.8)
axes[0].set_title("Top 15 coefficienti Ridge (feature standardizzate)")
axes[0].set_xlabel("coefficiente (effetto su log-prezzo)")

axes[1].barh(perm_imp.index, perm_imp.values, color="#2980b9")
axes[1].set_title("Top 15 permutation importance (test)")
axes[1].set_xlabel("aumento RMSE log se mescolata")
plt.tight_layout(); plt.show()

In [ ]:
# Correzione di Duan (smearing): il fattore S si stima sui residui di TRAINING (mai sul test)
res_train_log = y_train - pd.Series(best_model.predict(X_train), index=X_train.index)
S = np.mean(np.exp(res_train_log))

pred_log_test = pd.Series(best_model.predict(X_test), index=X_test.index)
true_d      = np.expm1(y_test)
pred_naive  = np.expm1(pred_log_test)                  # back-transform standard (mediana)
pred_duan   = S * (np.expm1(pred_log_test) + 1) - 1    # corretto per la media

print(f"Fattore di smearing S = {S:.5f}  (>1 -> compensa la sottostima)\n")
print(f"{'':<10}{'residuo medio $':>18}{'MAE $':>12}")
print(f"{'naive':<10}{(true_d-pred_naive).mean():>18,.0f}{mean_absolute_error(true_d, pred_naive):>12,.0f}")
print(f"{'Duan':<10}{(true_d-pred_duan).mean():>18,.0f}{mean_absolute_error(true_d, pred_duan):>12,.0f}")
print("\nLo smearing riduce il bias medio; la precisione puntuale (MAE) resta sostanzialmente invariata.")

In [ ]:
import joblib

# Template di valori tipici dal training: riempie le feature non fornite dall'utente
_feature_template = {
    col: (X_train[col].median() if pd.api.types.is_numeric_dtype(X_train[col])
          else X_train[col].mode()[0])
    for col in X_train.columns
}

def predict_price(input_dict: dict) -> float:
    """Stima il prezzo di vendita ($) da un dizionario di caratteristiche.

    Le feature non fornite usano valori tipici del training set. La funzione
    esegue l'INTERA pipeline addestrata (preprocessing + modello), quindi un dato
    nuovo riceve esattamente lo stesso trattamento dei dati di training.
    """
    row = {**_feature_template, **input_dict}
    X_new = pd.DataFrame([row])[X_train.columns]      # stesso schema e ordine del training
    pred_log = best_model.predict(X_new)[0]
    return float(np.expm1(pred_log))                  # back-transform: da log a dollari


# --- Esempi d'uso ---
esempi = {
    "Casa grande, qualita' alta, quartiere pregiato":
        {"Gr Liv Area": 2000, "Overall Qual": 8, "Year Built": 2005,
         "Neighborhood": "NridgHt", "Garage Cars": 2, "Total Bsmt SF": 1000},
    "Casa piccola, vecchia, quartiere economico":
        {"Gr Liv Area": 900, "Overall Qual": 4, "Year Built": 1950,
         "Neighborhood": "OldTown", "Garage Cars": 0, "Total Bsmt SF": 500},
    "Solo valori tipici (nessun input)": {},
}
for descrizione, caratteristiche in esempi.items():
    print(f"  {descrizione:48s} -> ${predict_price(caratteristiche):,.0f}")

In [ ]:
# Riproducibilita': salviamo la pipeline completa e la ricarichiamo
joblib.dump(best_model, "ames_best_model.joblib")
modello_caricato = joblib.load("ames_best_model.joblib")

uguale = np.isclose(modello_caricato.predict(X_test.iloc[:1])[0],
                    best_model.predict(X_test.iloc[:1])[0])
print("Pipeline salvata in 'ames_best_model.joblib' e ricaricata.")
print("La predizione del modello ricaricato e' identica all'originale:", bool(uguale))
print("\nNota: per ricaricare il modello in una NUOVA sessione, le classi custom")
print("(SkewedLogTransformer, DropDuplicateColumns) devono essere definite nell'ambiente.")
print("Il seed fisso (RANDOM_STATE = 42) rende l'intero notebook riproducibile.")